1. Project Overview
The project, “**Deep Learning-Based Real-Time Fire and Smoke Detection Using Surveillance Video**,” uses a pretrained YOLO object detection model to identify fire and smoke in images and videos. The system displays detected objects with bounding boxes and provides a browser-based interface for uploading files and viewing the results.

**Section 1:**
**Install Required Libraries**
The code installs ultralytics and huggingface_hub. Ultralytics is used for YOLO object detection, while Hugging Face Hub downloads the pretrained model weights.

In [ ]:
!pip install -q ultralytics huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 628.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 4.8 MB/s eta 0:00:00


Section 2: Load the Pretrained YOLO Model
The code downloads best.pt from the Hugging Face repository nobita111222/fire-smoke-yolov8n and loads it using YOLO(weights). This model is used to recognize fire and smoke.

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

weights = hf_hub_download(
    repo_id="nobita111222/fire-smoke-yolov8n",
    filename="best.pt"
)

model = YOLO(weights)

print(model.names)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


best.pt: reconstructing file:   0%|          |  0.00B / 6.23MB            

best.pt: downloading bytes:           |  0.00B            

{0: 'smoke', 1: 'fire'}


Section 3: Image Detection
The user uploads an image. The YOLO model processes it with a confidence threshold of 0.25. The detected objects are marked with bounding boxes and displayed as an output image.

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))

results = model.predict(
    source=image_path,
    conf=0.25,
    save=True
)

display(results[0].plot())

Saving AI Fire and Smoke Detection Demo.png to AI Fire and Smoke Detection Demo.png

image 1/1 /content/AI Fire and Smoke Detection Demo.png: 448x640 1 smoke, 1 fire, 306.4ms
Speed: 23.5ms preprocess, 306.4ms inference, 29.4ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict


array([[[238, 200, 168],
        [242, 203, 168],
        [242, 203, 168],
        ...,
        [245, 217, 190],
        [245, 218, 191],
        [242, 215, 188]],

       [[241, 202, 168],
        [242, 203, 166],
        [243, 204, 167],
        ...,
        [244, 216, 189],
        [245, 216, 189],
        [242, 215, 189]],

       [[243, 205, 169],
        [242, 203, 165],
        [243, 203, 166],
        ...,
        [245, 215, 188],
        [244, 216, 189],
        [244, 216, 190]],

       ...,

       [[126, 121, 119],
        [126, 120, 116],
        [127, 122, 119],
        ...,
        [ 45,  39,  33],
        [ 49,  44,  37],
        [ 46,  41,  36]],

       [[127, 123, 120],
        [125, 119, 116],
        [127, 123, 119],
        ...,
        [ 46,  40,  35],
        [ 49,  44,  38],
        [ 48,  43,  39]],

       [[125, 119, 120],
        [127, 123, 121],
        [127, 123, 120],
        ...,
        [ 48,  41,  39],
        [ 47,  40,  37],
        [ 49,  43,  39]]

Section 4: Video Detection
The uploaded video is processed by YOLO. The model detects fire and smoke in video frames, draws bounding boxes, and saves the processed output. The notebook then attempts to display the processed video.

In [ ]:
from google.colab import files
from IPython.display import Video, display
from pathlib import Path

uploaded = files.upload()
video_path = next(iter(uploaded))

results = model.predict(
    source=video_path,
    conf=0.25,
    save=True
)

output_path = Path(results[0].save_dir) / Path(video_path).name

if output_path.exists():
    display(Video(str(output_path), embed=True))
else:
    print("Check the output video in the runs/detect folder.")

Saving 204361-924209264.mp4 to 204361-924209264.mp4

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/495) /content/204361-924209264.mp4: 384x640 1 smoke, 80.8ms
video 1/1 (frame 2/495) /content/204361-924209264.mp4: 384x640 1 smoke, 78.1ms
video 1/1 (frame 3/495) /content/204361-924209264.mp4: 384x640 1 smoke, 94.4ms
video 1/1 (frame 4/495) /content/204361-924209264.mp4: 384x640 1 smoke, 77.1ms
video 1/1 (frame 5/495) /content/204361-924209264.mp4: 384x640 1 smoke, 75.3ms
video 1/1 (f

Section 5: Model File Management
The code searches /content for .pt model files and copies the downloaded weights to /content/best.pt. This helps verify and keep a local copy of the model.

In [ ]:

import os

model_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".pt"):
            model_files.append(os.path.join(root, file))

print("YOLO model files found:")

for path in model_files:
    print(path)

if not model_files:
    print("No local .pt model found.")


YOLO model files found:
No local .pt model found.


In [ ]:

from huggingface_hub import hf_hub_download
import shutil
import os

weights = hf_hub_download(
    repo_id="nobita111222/fire-smoke-yolov8n",
    filename="best.pt"
)

model_path = "/content/best.pt"
shutil.copy2(weights, model_path)

print("Model saved at:", model_path)
print("File exists:", os.path.exists(model_path))


Model saved at: /content/best.pt
File exists: True


Section 6: Create the Web Interface Using Streamlit
The code creates app.py, which builds the browser interface. Users can upload images or videos, click Start Detection, view detection results, and download the processed video. The model is cached so it does not need to be loaded again for every interaction.

In [ ]:

%%writefile app.py
import streamlit as st
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import tempfile
import os

st.set_page_config(page_title="Fire & Smoke Detection", page_icon="🔥")

st.title("🔥 Fire and Smoke Detection")
st.write("Upload an image or video to detect fire and smoke using YOLO.")

@st.cache_resource
def load_model():
    weights = hf_hub_download(
        repo_id="nobita111222/fire-smoke-yolov8n",
        filename="best.pt"
    )
    return YOLO(weights)

model = load_model()

uploaded_file = st.file_uploader(
    "Upload image or video",
    type=["jpg", "jpeg", "png", "mp4", "avi", "mov"]
)

if uploaded_file is not None:
    is_video = uploaded_file.type.startswith("video")

    suffix = os.path.splitext(uploaded_file.name)[1]

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp:
        temp.write(uploaded_file.getvalue())
        input_path = temp.name

    st.write("File uploaded:", uploaded_file.name)

    if st.button("Start Detection"):
        with st.spinner("Detecting fire and smoke..."):
            results = model.predict(
                source=input_path,
                conf=0.25,
                save=True,
                stream=False
            )

            if is_video:
                output_dir = results[0].save_dir
                output_path = os.path.join(
                    output_dir, os.path.basename(input_path)
                )

                if os.path.exists(output_path):
                    st.video(output_path)
                else:
                    st.warning("Processed video was not found.")
            else:
                st.image(results[0].plot(), channels="BGR")

            detected = set()
            for result in results:
                for box in result.boxes:
                    detected.add(model.names[int(box.cls[0])])

            if detected:
                st.error("Detected: " + ", ".join(sorted(detected)))
            else:
                st.success("No fire or smoke detected.")

    os.unlink(input_path)


Writing app.py


In [ ]:

%pip install -q streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.6 MB/s eta 0:00:00


In [ ]:

import os

print("app.py exists:", os.path.exists("/content/app.py"))
print("Current folder:", os.getcwd())


app.py exists: True
Current folder: /content


In [ ]:
%pip install -q streamlit pyngrok

In [ ]:
import os

print("App file exists:", os.path.exists("/content/app.py"))

App file exists: True


In [ ]:
%pip install -q streamlit pyngrok

In [ ]:
%%writefile /content/app.py
import streamlit as st
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import tempfile
import os
import glob

st.set_page_config(page_title="Fire & Smoke Detection", page_icon="🔥")
st.title("🔥 Fire and Smoke Detection")
st.write("Upload an image or video to detect fire and smoke using YOLO.")

@st.cache_resource
def load_model():
    weights = hf_hub_download(
        repo_id="nobita111222/fire-smoke-yolov8n",
        filename="best.pt"
    )
    return YOLO(weights)

model = load_model()

uploaded_file = st.file_uploader(
    "Upload image or video",
    type=["jpg", "jpeg", "png", "mp4", "avi", "mov"]
)

if uploaded_file is not None:
    is_video = uploaded_file.type.startswith("video")
    suffix = os.path.splitext(uploaded_file.name)[1]

    if st.button("Start Detection"):
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp:
            temp.write(uploaded_file.getvalue())
            input_path = temp.name

        try:
            with st.spinner("Detecting fire and smoke..."):
                results = model.predict(
                    source=input_path,
                    conf=0.25,
                    save=True
                )

                detected = set()
                for result in results:
                    for box in result.boxes:
                        detected.add(model.names[int(box.cls[0])])

                if detected:
                    st.error("Detected: " + ", ".join(sorted(detected)))
                else:
                    st.success("No fire or smoke detected.")

                if is_video:
                    output_dir = str(results[0].save_dir)
                    candidates = []
                    for ext in ["mp4", "avi", "mov", "mkv"]:
                        candidates += glob.glob(
                            os.path.join(output_dir, f"*.{ext}")
                        )

                    if candidates:
                        output_path = max(candidates, key=os.path.getmtime)
                        st.video(output_path)
                        with open(output_path, "rb") as f:
                            st.download_button(
                                "Download Processed Video",
                                f,
                                file_name="fire_smoke_detected.mp4",
                                mime="video/mp4"
                            )
                    else:
                        st.warning("Processed video not found. Check Colab output folder.")
                else:
                    st.image(results[0].plot(), channels="BGR")
        finally:
            os.unlink(input_path)

Overwriting /content/app.py


In [ ]:
from getpass import getpass
from pyngrok import ngrok
import subprocess
import time

token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

process = subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true"
])

time.sleep(5)

public_url = ngrok.connect(8501).public_url
print("Open your UI here:", public_url)

Enter your ngrok authtoken: ··········
Open your UI here: https://persevere-collision-chewing.ngrok-free.dev


**Working Flow of the Project**

Upload Image or Video

Load Pretrained YOLO Model

Detect Fire and Smoke

Display Bounding Boxes and Results

View or Download Processed Output